## Shelters and Reachability

Attach shelters and validate the graph connectivity and reachability

In [2]:
import osmnx as ox
import pandas as pd
import geopandas as gpd
import networkx as nx

In [11]:
def load_shelters(csv_path, default_capacity=750):
    """
    Load shelters from CSV and return GeoDataFrame (EPSG:4326).
    Expected columns: name, lat, lon, capacity
    """
    shelters = pd.read_csv(csv_path)

    required_cols = {"name", "lat", "lon"}
    missing = required_cols - set(shelters.columns)
    if missing:
        raise ValueError(f"Shelter file missing columns: {missing}")

    shelters["lat"] = pd.to_numeric(shelters["lat"], errors="coerce")
    shelters["lon"] = pd.to_numeric(shelters["lon"], errors="coerce")
    shelters = shelters.dropna(subset=["lat", "lon"])

    if "capacity" not in shelters.columns:
        shelters["capacity"] = default_capacity
    else:
        shelters["capacity"] = shelters["capacity"].fillna(default_capacity).astype(int)

    shelters_gdf = gpd.GeoDataFrame(
        shelters,
        geometry=gpd.points_from_xy(shelters["lon"], shelters["lat"]),
        crs="EPSG:4326",
    )

    return shelters_gdf

In [13]:
shelters_gdf = load_shelters("../data/raw/shelter/hatyai_shelters.csv")
print(f"[Shelters] Loaded {len(shelters_gdf)} valid shelters")
print(shelters_gdf[["name", "capacity"]].to_string())
shelters_gdf.to_file(
    "../data/processed/hatyai_shelters.geojson",
    driver="GeoJSON"
)

[Shelters] Loaded 5 valid shelters
                                     name  capacity
0     โรงเรียนเทศบาล 1 (เอ็งเสียงสามัคคี)       750
1          โรงเรียนเทศบาล 2 (บ้านหาดใหญ่)       750
2  โรงเรียนเทศบาล 3 (โศภนพิทยาคุณานุสรณ์)       750
3         โรงเรียนเทศบาล 4 (วัดคลองเรียน)       750
4     โรงเรียนนานาชาติเซาท์เทิร์น หาดใหญ่       750


In [17]:
def attach_shelters_to_nodes(graph_proj, shelters_gdf):
    """
    Snap shelters to nearest OSMnx nodes using OSMnx native method.
    """
    shelters_proj = shelters_gdf.to_crs(graph_proj.graph["crs"])
    xs = shelters_proj.geometry.x
    ys = shelters_proj.geometry.y

    nearest_nodes = ox.distance.nearest_nodes(graph_proj, X=xs, Y=ys)

    shelter_map = {}
    for node_id, name, capacity in zip(
        nearest_nodes, 
        shelters_gdf["name"], 
        shelters_gdf["capacity"]
    ):
        shelter_map.setdefault(node_id, []).append((name, capacity))

    nodes, edges = ox.graph_to_gdfs(graph_proj)

    nodes["is_shelter"] = False
    nodes["shelter_names"] = ""
    nodes["shelter_capacity"] = 0.0

    for node_id, shelters in shelter_map.items():
        if node_id in nodes.index:
            names = [s[0] for s in shelters]
            capacities = [s[1] for s in shelters]

            nodes.loc[node_id, "is_shelter"] = True
            nodes.loc[node_id, "shelter_names"] = "; ".join(sorted(set(names)))
            nodes.loc[node_id, "shelter_capacity"] = sum(capacities)

    return nodes, edges

In [18]:
graph = ox.load_graphml("../data/processed/hatyai_graph_with_capacity.graphml")

graph_proj = ox.project_graph(graph)
nodes, edges = attach_shelters_to_nodes(graph_proj, shelters_gdf)
print(f"[Shelters] Attached to {nodes['is_shelter'].sum()} nodes")
graph_updated  = ox.graph_from_gdfs(nodes, edges)

# Project back to lat/lon before saving
graph_updated = ox.project_graph(graph_updated, to_crs="EPSG:4326")
ox.save_graphml(graph_updated, "../data/processed/hatyai_graph_with_shelters.graphml")

[Shelters] Attached to 5 nodes


Optionally mirror edges to allow reverse travel in reachability checks

In [20]:
MAKE_EDGES_BIDIR = True

edges_reset = edges.reset_index()
if MAKE_EDGES_BIDIR:
    reversed_edges = edges_reset.copy()
    reversed_edges[["u", "v"]] = reversed_edges[["v", "u"]]

    edges_use = (
        pd.concat([edges_reset, reversed_edges], ignore_index=True)
        .drop_duplicates(subset=["u", "v", "key"])
    )
else:
    edges_use = edges_reset

print("edges_use rows:", len(edges_use))

edges_use rows: 8876


In [21]:
def check_static_connectivity(nodes_df, edges_df):
    """
    Check structural connectivity of the graph.
    """

    G = nx.from_pandas_edgelist(
        edges_df,
        source="u",
        target="v",
        create_using=nx.DiGraph()
    )

    if G.number_of_nodes() == 0:
        print("[WARNING] Graph is empty.")
        return None

    G_u = G.to_undirected()

    n_comp = nx.number_connected_components(G_u)
    largest = max(nx.connected_components(G_u), key=len)
    coverage = len(largest) / G.number_of_nodes()

    print("------ Connectivity Report ------")
    print(f"Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()}")
    print(f"Connected components: {n_comp}")
    print(f"Largest component coverage: {coverage:.3f}")

    isolates = list(nx.isolates(G))
    print(f"Isolates: {len(isolates)}")

    return {
        "n_components": n_comp,
        "coverage": coverage,
        "isolates": isolates
    }

In [22]:
def check_reachability_to_shelters(nodes_df, edges_df):
    """
    Check if all nodes can reach at least one shelter.
    """

    if "is_shelter" not in nodes_df.columns:
        raise ValueError("Nodes must contain 'is_shelter' column")

    shelter_nodes = set(nodes_df.index[nodes_df["is_shelter"]])

    if not shelter_nodes:
        print("[WARNING] No shelters found in nodes.")
        return None

    G_dir = nx.from_pandas_edgelist(
        edges_df,
        source="u",
        target="v",
        edge_attr="length",
        create_using=nx.DiGraph(),
    )

    G_rev = G_dir.reverse(copy=False)

    dist = nx.multi_source_dijkstra_path_length(
        G_rev,
        list(shelter_nodes),
        weight="length"
    )

    reachable = set(dist.keys())
    all_nodes = set(nodes_df.index)
    unreachable = all_nodes - reachable

    print("------ Reachability Report ------")
    print(f"Shelters: {len(shelter_nodes)}")
    print(f"Reachable nodes: {len(reachable)}")
    print(f"Unreachable nodes: {len(unreachable)}")

    return {
        "reachable": reachable,
        "unreachable": unreachable,
        "distances": dist
    }

In [23]:
check_static_connectivity(nodes, edges_use)
check_reachability_to_shelters(nodes, edges_use)

------ Connectivity Report ------
Nodes: 3412 | Edges: 8835
Connected components: 1
Largest component coverage: 1.000
Isolates: 0
------ Reachability Report ------
Shelters: 5
Reachable nodes: 3412
Unreachable nodes: 0


{'reachable': {2506096642,
  8634671109,
  2506096652,
  8643084306,
  485580819,
  485580818,
  485580824,
  2506088476,
  485580829,
  2506096670,
  2052497441,
  13472858147,
  2506088483,
  1680662566,
  485580840,
  485580841,
  485580842,
  2506088491,
  2506088492,
  485580845,
  8932024360,
  8932024367,
  1680662578,
  8932532276,
  8932532279,
  1680662584,
  8932532280,
  2506088507,
  13378469948,
  2506088509,
  13378469950,
  1680662592,
  2506096706,
  4900464590,
  2506088518,
  8364261453,
  8364261455,
  2506088527,
  2506088529,
  2506088528,
  2506088533,
  2506088535,
  1680662616,
  2506088538,
  13347725404,
  13347725405,
  1680662622,
  1680662629,
  617234534,
  617234535,
  8932245608,
  8359542886,
  8359542888,
  8359542887,
  8642314348,
  8359542889,
  1407221870,
  2506088559,
  1680662647,
  2506096763,
  11055915144,
  11055915145,
  1680662668,
  1680663828,
  2506088592,
  617234576,
  1680662672,
  1680662680,
  8642592922,
  8642592923,
  168066268